# Voy a bajar datos de la wikipedia de datos historicos


In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import unicodedata
import numpy as np
import unicodedata
pd.set_option("display.max_rows", 1000)
pd.set_option("display.max_columns", 1000)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

url ="https://es.wikipedia.org/wiki/Anexo:C%C3%B3digos_del_COI"
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9",
}

response = requests.get(url, headers=headers)

if response.ok:
    print("OK → status 200-299")
else:
    print("Error →", response.status_code)


OK → status 200-299


In [2]:
     # Vamos a parsearlo 

soup = BeautifulSoup(response.text, "lxml")    # Toma el HTML crudo (response.text):
                                                    # Usa el parser lxml
                                                         # lxml es:
                                                         # más rápido
                                                         # más estricto
                                                         # más estable
                                                         # repara HTML roto mejor que html.parser
                                                    # Convierte el HTML en un árbol DOM navegable
                                                    # El HTML pasa de ser texto plano a ser un árbol de objetos:
                                                          # Código
                                                          # html
                                                          #  └── body
                                                          #       └── h1
                                                          #            └── "Hola"
                                                    # Crea el objeto soup
                                                         # soup es un árbol completo del HTML, donde puedes hacer:
                                                         # soup.find("table")
                                                         # soup.find_all("a")
                                                         # soup.select(".price")
                                                         # soup.get_text()

In [3]:
print(bool(soup))  # Para saber si no esta vacio


True


In [4]:
# Comprobar si hay tables 
tables = soup.find_all("table")                 # busca TODAS las etiquetas <table> del HTML
print(f"Total tablas; {len(tables)}")           # Esto NO significa:
                                                    # que haya 10 DataFrames
                                                    # que pandas pueda leerlas
                                                    # que todas sean útiles
                                                    # Solo significa: 
                                                    # 👉 El HTML contiene 13 elementos <table>



Total tablas; 5


In [5]:
def tiene_cabeceras(tabla):
    """Debe tener <th> para ser tabla real."""
    return tabla.find("th") is not None

def tiene_filas(tabla):
    """Debe tener varias filas <tr>."""
    filas = tabla.find_all("tr")
    return len(filas) >= 2

def tiene_columnas(tabla):
    """Debe tener varias columnas en la primera fila."""
    primera = tabla.find("tr")
    if not primera:
        return False
    celdas = primera.find_all(["td", "th"])
    return len(celdas) >= 2

def no_es_navbox(tabla):
    """Excluir tablas de navegación, infobox, sidebars, etc."""
    clase = tabla.get("class", [])
    basura = ["navbox", "infobox", "sidebar", "metadata", "vertical-navbox"]
    return not any(b in clase for b in basura)

def tiene_texto_util(tabla):
    """Debe tener texto significativo."""
    texto = tabla.get_text(" ", strip=True)
    return len(texto.split()) > 5

In [6]:
tablas = soup.find_all("table")
print(f"Tablas encontradas: {len(tablas)}")

tablas_validas = []
for t in tablas:
    if (
        tiene_cabeceras(t)
        and tiene_filas(t)
        and tiene_columnas(t)
        and no_es_navbox(t)
        and tiene_texto_util(t)
    ):
        tablas_validas.append(t)

print(f"Tablas válidas detectadas: {len(tablas_validas)}")


Tablas encontradas: 5
Tablas válidas detectadas: 2



Saco las tablas validas

In [7]:
dfs = []

for t in tablas_validas:
    rows = []
    for tr in t.find_all("tr"):
        cells = tr.find_all(["th", "td"])
        cells = [c.get_text(" ", strip=True) for c in cells]
        if cells:
            rows.append(cells)

    # Saltar tablas sin contenido real
    if len(rows) < 2:
        continue

    # Normalizar columnas (rellenar filas cortas)
    max_cols = max(len(r) for r in rows)
    rows = [r + [""] * (max_cols - len(r)) for r in rows]

    # Crear DataFrame seguro
    df = pd.DataFrame(rows[1:], columns=rows[0]) # va generando uno a unu
    dfs.append(df)   # los añade



In [8]:
print(f" Numero de dataframes {len(dfs)}")
print("-----------------------------")
for i, df in enumerate(dfs):
    print(f"dataframe:{i} {df.shape} {df.columns.tolist()}")
    print(df.head(1))

    print("--------------------")


 Numero de dataframes 2
-----------------------------
dataframe:0 (280, 5) ['Código', 'Federación nacional', 'desde', 'hasta', 'Notas']
         Código Federación nacional desde hasta Notas
0  A [ editar ]                                      
--------------------
dataframe:1 (3, 4) ['Código', 'Federación nacional', 'Año', 'Notas']
         Código         Federación nacional   Año  \
0  IOA [ n. 1 ]  Atleta Olímpico Individual  2000   

                                                                              Notas  
0  Usado por Timor Oriental bajo el periodo de administración de la ONU ( UNTAET ).  
--------------------


### Solo me interesa el dataframe 0 - Codigos antiguos

In [9]:

df_ico = dfs[0]

df_ico.tail(20)       

,Código,Federación nacional,desde,hasta,Notas
260,UZB,Uzbekistán Uzbekistán Uzbekistán,1994,,1900-1912 – Perteneciente a RUS 1924-1988 – Perteneciente a URS 1992 – Perteneciente a EUN
261,V [ editar ],,,,
262,VAN,Vanuatu Vanuatu Vanuatu,1988,,
263,VEN,Venezuela Venezuela,1948,,
264,VIE,Vietnam Vietnam Vietnam,1952,,"1956-1972 – Usado por Vietnam del Sur , sin representación de Vietnam del Norte ."
265,VIN,San Vicente y las Granadinas San Vicente y las Granadinas San Vicente y las Granadinas,1988,,1960 – Perteneciente a BWI
266,VOL,Alto Volta Alto Volta,1972,1984,Actualmente como BUR
267,X [ editar ],,,,
268,XXB,Equipo Mixto,1896,1904,"Código para todos los equipos cuyos atletas provienen de diferentes países. Fue usado solamente en los primeros juegos olímpicos, ya que no existían en ese entonces los comités olímpicos nacionales. Fue cambiado de ZZX a XXB en 2024."
269,Y [ editar ],,,,


In [10]:
df_ico= df_ico.drop(columns=["Notas"])

df_ico.head()

,Código,Federación nacional,desde,hasta
0,A [ editar ],,,
1,AFG,AFG Afganistán,1936,
2,AHO,Antillas Neerlandesas Antillas Neerlandesas,1960,2008
3,AIN,Atletas Individuales Neutrales,2024,2026
4,ALB,Albania Albania,1972,


In [11]:
def tipos_por_columna(df):
    for col in df.columns:
        print(f"\n--- {col} ---")
        print(df[col].apply(lambda x: type(x).__name__).value_counts())

tipos_por_columna(df_ico)


--- Código ---
Código
str    280
Name: count, dtype: int64

--- Federación nacional ---
Federación nacional
str    280
Name: count, dtype: int64

--- desde ---
desde
str    280
Name: count, dtype: int64

--- hasta ---
hasta
str    280
Name: count, dtype: int64


Separo los valores codigos validos y estudio los no validos

In [12]:
df_validos = df_ico[df_ico["Código"].str.len() == 3]
df_no_validos = df_ico[df_ico["Código"].str.len() != 3]


In [13]:
df_validos.head()

,Código,Federación nacional,desde,hasta
1,AFG,AFG Afganistán,1936,
2,AHO,Antillas Neerlandesas Antillas Neerlandesas,1960,2008
3,AIN,Atletas Individuales Neutrales,2024,2026
4,ALB,Albania Albania,1972,
5,ALG,Argelia Argelia Argelia,1964,


Limpio de los vacios

In [14]:
df_validos_antiguos = df_validos[df_validos["hasta"].astype(str).str.strip() != ""]
print(df_validos_antiguos.info())
df_validos_antiguos.head()

<class 'pandas.DataFrame'>
Index: 41 entries, 2 to 277
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Código               41 non-null     str  
 1   Federación nacional  41 non-null     str  
 2   desde                41 non-null     str  
 3   hasta                41 non-null     str  
dtypes: str(4)
memory usage: 1.6 KB
None


,Código,Federación nacional,desde,hasta
2,AHO,Antillas Neerlandesas Antillas Neerlandesas,1960,2008
3,AIN,Atletas Individuales Neutrales,2024,2026
9,ANZ,Australasia ANZ Australasia,1908,1912
27,BIR,Birmania Birmania Birmania,1948,1988
30,BOH,Bohemia,1900,1912


In [15]:
df_no_validos.head(200)

,Código,Federación nacional,desde,hasta
0,A [ editar ],,,
17,B [ editar ],,,
40,C [ editar ],,,
66,D [ editar ],,,
72,E [ editar ],,,
83,F [ editar ],,,
89,G [ editar ],,,
105,H [ editar ],,,
112,I [ editar ],,,
116,IOP [ n. 1 ],Participante Olímpico Independiente,1992,1992


In [16]:
df_no_validos=df_no_validos[~df_no_validos["Código"].astype(str).str.contains("editar", na=False)]

df_no_validos

,Código,Federación nacional,desde,hasta
116,IOP [ n. 1 ],Participante Olímpico Independiente,1992,1992
203,Comité Olímpico Ruso,2020,2022,"Usado por Rusia , por la suspensión de su comité."
273,Reino de Yugoslavia YUG,Reino de Yugoslavia Reino de Yugoslavia,1920,1936
274,R.F.S. de Yugoslavia R.F.S. de Yugoslavia,1954,1988,"Equipos consecutivos: BIH , CRO , MKD , MNE y SLO"
275,R.F. de Yugoslavia,1996,2002,Equipo consecutivo: SCG


Voy a desplazat las otras filas 


In [17]:
indices = [1, 2,3, 4]
for x in indices:
    df_no_validos.iloc[x, 1:] = df_no_validos.iloc[x, :-1].values
    df_no_validos.iloc[x, 0] = None
df_no_validos

,Código,Federación nacional,desde,hasta
116,IOP [ n. 1 ],Participante Olímpico Independiente,1992,1992
203,NaN,Comité Olímpico Ruso,2020,2022
273,NaN,Reino de Yugoslavia YUG,Reino de Yugoslavia Reino de Yugoslavia,1920
274,NaN,R.F.S. de Yugoslavia R.F.S. de Yugoslavia,1954,1988
275,NaN,R.F. de Yugoslavia,1996,2002


Cambiare a mano las 4 que faltan que no salen bien de la web pero alli se ven bien

In [18]:
valores = ['IOP','ROC', 'YUG', 'YUG', 'YUG']
indices = [0,1,2,3,4]

for i, idx in enumerate(indices):
    df_no_validos.iloc[idx, 0] = valores[i]


df_no_validos

,Código,Federación nacional,desde,hasta
116,IOP,Participante Olímpico Independiente,1992,1992
203,ROC,Comité Olímpico Ruso,2020,2022
273,YUG,Reino de Yugoslavia YUG,Reino de Yugoslavia Reino de Yugoslavia,1920
274,YUG,R.F.S. de Yugoslavia R.F.S. de Yugoslavia,1954,1988
275,YUG,R.F. de Yugoslavia,1996,2002


In [19]:
df_no_validos.loc[273, "desde"] = '1900'
df_no_validos

,Código,Federación nacional,desde,hasta
116,IOP,Participante Olímpico Independiente,1992,1992
203,ROC,Comité Olímpico Ruso,2020,2022
273,YUG,Reino de Yugoslavia YUG,1900,1920
274,YUG,R.F.S. de Yugoslavia R.F.S. de Yugoslavia,1954,1988
275,YUG,R.F. de Yugoslavia,1996,2002


Ahora rejunto las tablas 

In [20]:
df_ico_antiguo = pd.concat([df_validos, df_no_validos], ignore_index=True)
df_ico_antiguo.head()

,Código,Federación nacional,desde,hasta
0,AFG,AFG Afganistán,1936,
1,AHO,Antillas Neerlandesas Antillas Neerlandesas,1960,2008
2,AIN,Atletas Individuales Neutrales,2024,2026
3,ALB,Albania Albania,1972,
4,ALG,Argelia Argelia Argelia,1964,


Eliminamos valores antiguos en la tercera columna

In [21]:
df_ico_antiguo["Federación nacional"] = df_ico_antiguo["Federación nacional"].apply(
    lambda x: " ".join(dict.fromkeys(str(x).split()))
)
df_ico_antiguo.head()

,Código,Federación nacional,desde,hasta
0,AFG,AFG Afganistán,1936,
1,AHO,Antillas Neerlandesas,1960,2008
2,AIN,Atletas Individuales Neutrales,2024,2026
3,ALB,Albania,1972,
4,ALG,Argelia,1964,


Y nos quedamos solo con los que tengan fecha en hasta que son los que buscabamos

In [22]:
df_ico_antiguo = df_ico_antiguo.replace(r"^\s*$", None, regex=True)
df_ico_antiguo = df_ico_antiguo.dropna(subset=["hasta"])

df_ico_antiguo.head(10)

,Código,Federación nacional,desde,hasta
1,AHO,Antillas Neerlandesas,1960,2008
2,AIN,Atletas Individuales Neutrales,2024,2026
8,ANZ,Australasia ANZ,1908,1912
25,BIR,Birmania,1948,1988
28,BOH,Bohemia,1900,1912
30,BOR,Borneo septentrional,1956,1956
37,BWI,Indias Occidentales,1960,1960
43,CEY,Ceilán,1948,1972
49,COB,Congo-Brazzaville,1964,1964
51,COK,Congo-Kinshasa,1968,1968


Ultimo paso poner nombres en ingles que lo voy a necesitar

In [23]:
from deep_translator import GoogleTranslator

df_ico_antiguo["Federación nacional"] = df_ico_antiguo["Federación nacional"].apply(
    lambda x: GoogleTranslator(source="auto", target="en").translate(str(x))
)

In [24]:
df_ico_antiguo.head(10)

,Código,Federación nacional,desde,hasta
1,AHO,Netherlands Antilles,1960,2008
2,AIN,Neutral Individual Athletes,2024,2026
8,ANZ,Australasia ANZ,1908,1912
25,BIR,Burma,1948,1988
28,BOH,Bohemia,1900,1912
30,BOR,North Borneo,1956,1956
37,BWI,West Indies,1960,1960
43,CEY,Ceylon,1948,1972
49,COB,Congo-Brazzaville,1964,1964
51,COK,Congo-Kinshasa,1968,1968


In [25]:
df_ico_antiguo.to_csv("Codigos_ICO_Antiguos.csv", index=False, encoding="utf-8")